# Admin config — T1 smoke notebook

Re-runnable end-to-end validation of the 5 admin endpoints + Redis hot reload
wired in R9 sandbox commits `e4cdd97d`..`06cd7300`.

**Pre-requisites** :
- Stack up (`.\scripts\start_stack.ps1`) with at least `api`, `postgres`, `redis`, `vol-engine` healthy
- Alembic migrations applied (start_stack does it automatically)
- `pip install requests` in the venv if not present

**Contract** : Restart & Run All should finish with every ✅ printed. A red
❌ means T1 regressed and the cell output points at the broken assertion.

**Scope NOT covered** : the React Settings page (manual browser check at
`http://localhost/settings`), and the MFA / auth layer (deliberately absent
in T1). Those are T3 polish items.

## Setup — shared helpers

In [ ]:
import json
import subprocess
import time

import requests

BASE = "http://localhost/api/v1/admin"


def get(path, **kw):
    r = requests.get(f"{BASE}{path}", timeout=5, **kw)
    return r


def put(path, body):
    return requests.put(f"{BASE}{path}", json=body, timeout=5)


def post(path, body):
    return requests.post(f"{BASE}{path}", json=body, timeout=5)


def vol_engine_logs(tail: int = 30) -> str:
    out = subprocess.run(
        ["docker", "compose", "logs", "vol-engine", "--tail", str(tail)],
        capture_output=True, text=True, check=False,
    )
    return out.stdout + out.stderr


def psql(query: str) -> str:
    out = subprocess.run(
        ["docker", "compose", "exec", "-T", "postgres",
         "psql", "-U", "fxvol", "-d", "fxvol", "-c", query],
        capture_output=True, text=True, check=False,
    )
    return out.stdout


print("helpers ready ; BASE =", BASE)

## Test 1 — GET /config returns the current version + full VolTradingConfig

Ce que tu dois voir : `version` ≥ 1 ; toutes les 8 sections (regime, signal,
sizing, exit_rules, surface, calibration, delta_hedge, structures) présentes
dans `config`. Une version=0 signifierait que la table est vide (migration
004 pas appliquée → re-run start_stack). 

In [ ]:
r = get("/config")
assert r.status_code == 200, r.text
body = r.json()
EXPECTED_SECTIONS = {"regime", "signal", "sizing", "exit_rules",
                     "surface", "calibration", "delta_hedge", "structures"}
assert EXPECTED_SECTIONS <= set(body["config"].keys())
print(f"✅ GET /config → 200 ; version={body['version']} ; 8 sections OK")

## Test 2 — GET /config/schema exports the full JSON Schema

Sortie attendue : un objet JSON Schema avec `properties` qui liste les 8
sections. C'est ce que le frontend RJSF consommera en T2 pour générer les
formulaires automatiquement.

In [ ]:
r = get("/config/schema")
assert r.status_code == 200
schema = r.json()
assert "properties" in schema
assert EXPECTED_SECTIONS <= set(schema["properties"].keys())
print(f"✅ schema has {len(schema['properties'])} top-level sections")

## Test 3 — PUT valid patch → new version + hot reload

(1) Le PUT doit renvoyer une version incrémentée et le patch appliqué.
(2) Peu après, vol-engine doit émettre `vol_engine_config_reloaded` dans
ses logs (via Redis pub/sub). Si le log n'apparaît pas, soit vol-engine
n'est pas abonné à `config:changed`, soit l'apply_config a levé silent.

In [ ]:
before = get("/config").json()["version"]
r = put("/config", {
    "patch": {"signal": {"threshold_vol_pts": 2.5, "model_p": "garch"}},
    "user": "smoke-notebook",
    "comment": "T1 smoke test 3",
})
assert r.status_code == 200, r.text
after = r.json()
assert after["version"] == before + 1
assert after["config"]["signal"]["threshold_vol_pts"] == 2.5
assert after["config"]["signal"]["model_p"] == "garch"
print(f"✅ PUT → version={after['version']} ; threshold=2.5 garch")

time.sleep(2.5)
logs = vol_engine_logs()
assert "vol_engine_config_reloaded threshold=2.500 model=garch" in logs, logs[-1500:]
print("✅ vol-engine hot reload confirmed in logs")

## Test 4 — PUT hors-bornes → 422 Pydantic error

Envoyer `threshold_vol_pts = 999.0` (hors de `[0.1, 10.0]`) doit déclencher
un 422 avec un détail Pydantic qui nomme le champ fautif. Transaction
annulée → pas de nouvelle version en DB.

In [ ]:
before = get("/config").json()["version"]
r = put("/config", {"patch": {"signal": {"threshold_vol_pts": 999.0}}})
assert r.status_code == 422
detail = r.json()["detail"]
assert "schema" in detail["message"]
assert any("threshold_vol_pts" in ".".join(map(str, e["loc"])) for e in detail["errors"])
assert get("/config").json()["version"] == before, "ROLLBACK KO : version bumped on a rejected PUT"
print("✅ 422 avec Pydantic detail ; DB non mutée")

## Test 5 — Cross-field validator (monotonic z-thresholds) → 422

Les trois seuils z (arm < strong < extreme) sont validés par un validator
Pydantic. Un patch qui inverse arm/strong doit renvoyer 422.

In [ ]:
r = put("/config", {"patch": {"signal": {"z_threshold_arm": 2.5, "z_threshold_strong": 1.8}}})
assert r.status_code == 422
payload_str = json.dumps(r.json())
assert "z_threshold_strong must be" in payload_str
print("✅ cross-field validator rejects non-monotonic z-thresholds")

## Test 6 — GET /config/history newest-first

Au moins 2 entrées (seed v1 + test 3). Order descending by version.

In [ ]:
r = get("/config/history", params={"limit": 20})
assert r.status_code == 200
hist = r.json()
assert len(hist) >= 2
versions = [row["version"] for row in hist]
assert versions == sorted(versions, reverse=True), versions
print(f"✅ history returned {len(hist)} rows, descending order : {versions[:5]}...")

## Test 7 — Revert to version 1 → new version + config restored + hot reload

Crée une nouvelle version (N+1) dont le payload dupliquait version=1 (seed,
defaults Pydantic). Vol-engine doit aussi recevoir le reload.

In [ ]:
r = post("/config/revert/1", {"user": "smoke-notebook", "comment": "T1 smoke revert to v1"})
assert r.status_code == 200, r.text
reverted = r.json()
assert reverted["config"]["signal"]["threshold_vol_pts"] == 1.0
assert reverted["config"]["signal"]["model_p"] == "har"
print(f"✅ revert → version={reverted['version']} ; defaults restored")

time.sleep(2.5)
logs = vol_engine_logs()
assert "vol_engine_config_reloaded threshold=1.000 model=har" in logs
print("✅ vol-engine hot reload after revert confirmed")

## Test 8 — Revert to an unknown version → 404

In [ ]:
r = post("/config/revert/9999", {})
assert r.status_code == 404
assert "9999 not found" in r.json()["detail"]
print("✅ 404 sur version inconnue")

## Test 9 — DB inspection : 1 row par événement, ordre cohérent

Requête SQL directe sur le container postgres. Affiche la history sous le
format `(version, updated_by, snippet de comment, threshold)`. Utile pour
vérifier que l'audit trail est complet.

In [ ]:
print(psql("SELECT version, updated_by, LEFT(comment, 40) AS comment_short, "
           "config->'signal'->>'threshold_vol_pts' AS threshold "
           "FROM vol_config ORDER BY version;"))

## Test 10 — Re-PUT post-revert produit une nouvelle version cohérente

In [ ]:
before = get("/config").json()["version"]
r = put("/config", {
    "patch": {"signal": {"threshold_vol_pts": 1.7}},
    "user": "smoke-notebook",
    "comment": "T1 smoke test 10 after revert",
})
assert r.status_code == 200
after = r.json()
assert after["version"] == before + 1
assert after["config"]["signal"]["threshold_vol_pts"] == 1.7
print(f"✅ re-PUT post-revert → version={after['version']}")

time.sleep(2.5)
logs = vol_engine_logs()
assert "vol_engine_config_reloaded threshold=1.700 model=har" in logs
print("✅ hot reload post-revert confirmed")

## Final — récap

Si tu vois 10 ✅ ci-dessus, T1 est validé end-to-end.

**À faire manuellement** (non couvert ici) :
- Ouvrir `http://localhost/settings` → formulaire visible, Save + Revert
  fonctionnent dans le navigateur

## Troubleshooting

| Symptôme | Cause probable | Fix |
|---|---|---|
| Test 1 `version=0` | Migration 004 pas appliquée | `docker compose exec api python -m alembic -c src/persistence/alembic.ini upgrade head` |
| Test 1 `ConnectionError` | API container down | `docker compose ps api` puis `docker compose up -d --build api` |
| Test 3 assert `config_reloaded` KO | vol-engine pas abonné | `docker compose logs vol-engine --tail 100` cherche `config_watcher_subscribe_failed` ; sinon `docker compose exec redis redis-cli PUBSUB NUMSUB config:changed` doit renvoyer 1 |
| Test 3 assert threshold incorrect | vol-engine image pas rebuild | `docker compose up -d --build vol-engine` |
| Test 4 passe avec 200 au lieu de 422 | api pas rebuild, Pydantic schema pas à jour | `docker compose up -d --build api` |
| Test 9 vide | postgres down ou table manquante | `docker compose exec postgres pg_isready` + `psql \d vol_config` |